# Module 01 — Train and Package an Iris Model for FastAPI

This notebook is the **offline build stage**. It trains, evaluates, packages, and reloads an Iris classifier before the FastAPI runtime is allowed to use it.

**Learning contract**

```text
Dataset → feature contract → split → Pipeline → evaluation → artifacts
                                                       │
                                             lifecycle boundary
                                                       ▼
                                               FastAPI serving
```

By the end you should be able to explain why training and serving are separate, what an artifact is, how feature schemas remain stable, and why the saved model must be reloaded and tested before deployment.

## 1. Runtime and reproducibility

The project uses Conda through `environment.yml`. We record the runtime versions used for this executed reference notebook.

In [1]:
import json, platform
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)

Python: 3.13.5
NumPy: 2.3.5
pandas: 2.2.3
scikit-learn: 1.8.0
joblib: 1.5.3


## 2. Stable project paths

The API reads from `artifacts/`; it does not depend on notebook memory.

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACT_DIR / "iris_model.joblib"
METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"
METRICS_PATH = ARTIFACT_DIR / "metrics.json"

print("Artifact directory:", ARTIFACT_DIR)

Artifact directory: .../01-iris-fastapi-local-serving/artifacts


## 3. Dataset → serving feature contract

The model and API must agree on **names, order, types, and units**. The public request fields are:

- `sepal_length_cm`
- `sepal_width_cm`
- `petal_length_cm`
- `petal_width_cm`

In [3]:
iris = load_iris(as_frame=True)
raw_df = iris.frame.copy()

FEATURE_NAMES = [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm",
]
CLASS_NAMES = [str(x) for x in iris.target_names]

X = raw_df.drop(columns=["target"]).copy()
X.columns = FEATURE_NAMES
y = raw_df["target"].copy()

print("Rows:", len(X))
print("Features:", FEATURE_NAMES)
print("Classes:", CLASS_NAMES)
print(X.head().to_string(index=False))

Rows: 150
Features: ['sepal_length_cm', 'sepal_width_cm', 'petal_length_cm', 'petal_width_cm']
Classes: ['setosa', 'versicolor', 'virginica']
 sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm
             5.1             3.5              1.4             0.2
             4.9             3.0              1.4             0.2
             4.7             3.2              1.3             0.2
             4.6             3.1              1.5             0.2
             5.0             3.6              1.4             0.2


## 4. Inspect assumptions before training

Deployment-minded ML starts by checking data properties rather than silently assuming them.

In [4]:
print("Missing values:", X.isna().sum().to_dict())
print("Feature minimums:", X.min().to_dict())
print("Feature maximums:", X.max().to_dict())
print("Class counts:", y.value_counts().sort_index().to_dict())

Missing values: {'sepal_length_cm': 0, 'sepal_width_cm': 0, 'petal_length_cm': 0, 'petal_width_cm': 0}
Feature minimums: {'sepal_length_cm': 4.3, 'sepal_width_cm': 2.0, 'petal_length_cm': 1.0, 'petal_width_cm': 0.1}
Feature maximums: {'sepal_length_cm': 7.9, 'sepal_width_cm': 4.4, 'petal_length_cm': 6.9, 'petal_width_cm': 2.5}
Class counts: {0: 50, 1: 50, 2: 50}


## 5. Reproducible train/test split

`random_state=42` makes the split deterministic; `stratify=y` preserves class balance.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train class counts:", y_train.value_counts().sort_index().to_dict())

Training rows: 120
Test rows: 30
Train class counts: {0: 40, 1: 40, 2: 40}


## 6. One deployable preprocessing + model Pipeline

```text
raw features → StandardScaler → LogisticRegression → prediction
```

Packaging preprocessing with the estimator reduces **training-serving skew**.

In [6]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=500, random_state=42)),
])

print(model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=500, random_state=42))])


## 7. Train offline

Training belongs in the build stage. `app.py` contains no `.fit()` call.

In [7]:
model.fit(X_train, y_train)
print("Training complete")

Training complete


## 8. Evaluate before packaging

A model should not become a deployment artifact without an explicit quality check.

In [8]:
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

Test accuracy: 0.9333
              precision    recall  f1-score   support

      setosa     1.0000    1.0000    1.0000        10
  versicolor     0.9000    0.9000    0.9000        10
   virginica     0.9000    0.9000    0.9000        10

    accuracy                         0.9333        30
   macro avg     0.9333    0.9333    0.9333        30
weighted avg     0.9333    0.9333    0.9333        30


In [9]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

Confusion matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]


## 9. Deployment gate

For this teaching lab, accuracy must be at least **0.90**. Real systems require domain-specific gates and approvals.

In [10]:
MINIMUM_ACCURACY = 0.90
assert accuracy >= MINIMUM_ACCURACY
print(f"PASS: {accuracy:.4f} >= {MINIMUM_ACCURACY:.2f}")

PASS: 0.9333 >= 0.90


## 10. Package operational metadata

The binary model is accompanied by its version, ordered feature schema, classes, build information, and evaluation evidence.

In [11]:
MODEL_VERSION = "1.0.0"

metadata = {
    "model_name": "iris-logistic-regression",
    "model_version": MODEL_VERSION,
    "framework": "scikit-learn",
    "scikit_learn_version": sklearn.__version__,
    "feature_names": FEATURE_NAMES,
    "feature_units": {name: "cm" for name in FEATURE_NAMES},
    "class_names": CLASS_NAMES,
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "random_state": 42,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

metrics = {
    "deployment_gate": {
        "metric": "accuracy",
        "minimum": MINIMUM_ACCURACY,
        "passed": True,
    },
    "accuracy": float(accuracy),
    "classification_report": classification_report(
        y_test, y_pred, target_names=CLASS_NAMES, output_dict=True
    ),
    "confusion_matrix": cm.tolist(),
}

print(json.dumps(metadata, indent=2))

{
  "model_name": "iris-logistic-regression",
  "model_version": "1.0.0",
  "framework": "scikit-learn",
  "scikit_learn_version": "1.8.0",
  "feature_names": [
    "sepal_length_cm",
    "sepal_width_cm",
    "petal_length_cm",
    "petal_width_cm"
  ],
  "feature_units": {
    "sepal_length_cm": "cm",
    "sepal_width_cm": "cm",
    "petal_length_cm": "cm",
    "petal_width_cm": "cm"
  },
  "class_names": [
    "setosa",
    "versicolor",
    "virginica"
  ],
  "training_rows": 120,
  "test_rows": 30,
  "random_state": 42
}


## 11. Serialize artifacts

`joblib` stores the fitted Pipeline; JSON keeps metadata and metrics inspectable.

In [12]:
joblib.dump(model, MODEL_PATH)
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

for path in (MODEL_PATH, METADATA_PATH, METRICS_PATH):
    print(f"{path.name}: {path.stat().st_size} bytes")

iris_model.joblib: 1889 bytes
model_metadata.json: 587 bytes
metrics.json: 1015 bytes


## 12. Reload from disk — the deployment boundary test

FastAPI will never receive this notebook's in-memory `model` variable. It loads the file, so we verify that exact boundary now.

In [13]:
loaded_model = joblib.load(MODEL_PATH)
loaded_metadata = json.loads(METADATA_PATH.read_text())
loaded_metrics = json.loads(METRICS_PATH.read_text())

assert loaded_metadata["feature_names"] == FEATURE_NAMES
assert loaded_metrics["deployment_gate"]["passed"] is True

print("Reloaded:", type(loaded_model).__name__, loaded_metadata["model_version"])

Reloaded: Pipeline 1.0.0


## 13. Simulate FastAPI's handoff

The HTTP layer validates JSON and then builds features in the order stored in metadata.

In [14]:
payload = {
    "sepal_length_cm": 5.1,
    "sepal_width_cm": 3.5,
    "petal_length_cm": 1.4,
    "petal_width_cm": 0.2,
}

ordered = loaded_metadata["feature_names"]
frame = pd.DataFrame(
    [[payload[name] for name in ordered]],
    columns=ordered,
)

print(frame.to_string(index=False))

 sepal_length_cm  sepal_width_cm  petal_length_cm  petal_width_cm
             5.1             3.5              1.4             0.2


In [15]:
class_id = int(loaded_model.predict(frame)[0])
probs = loaded_model.predict_proba(frame)[0]

result = {
    "prediction": CLASS_NAMES[class_id],
    "class_id": class_id,
    "probabilities": {
        name: round(float(p), 6)
        for name, p in zip(CLASS_NAMES, probs, strict=True)
    },
    "model_version": MODEL_VERSION,
}

assert result["prediction"] == "setosa"
assert abs(sum(result["probabilities"].values()) - 1.0) < 1e-5
print(json.dumps(result, indent=2))

{
  "prediction": "setosa",
  "class_id": 0,
  "probabilities": {
    "setosa": 0.980813,
    "versicolor": 0.019187,
    "virginica": 0.0
  },
  "model_version": "1.0.0"
}


## 14. What FastAPI receives

```text
iris_model.joblib      fitted scaler + classifier
model_metadata.json   feature schema + classes + version + provenance
metrics.json          evaluation evidence + deployment gate
```

Runtime flow:

```text
POST /predict → Pydantic → ordered DataFrame → loaded Pipeline → JSON response
```

`joblib` is suitable for this controlled educational workflow, but never load an untrusted pickle/joblib artifact.

## 15. Handoff checklist

- [x] data inspected
- [x] deterministic split
- [x] preprocessing + estimator packaged together
- [x] evaluation completed
- [x] deployment gate passed
- [x] artifacts serialized
- [x] serialized model reloaded
- [x] API-shaped input tested against the reloaded model

Next:

```bash
pytest -q
uvicorn app:app --host 127.0.0.1 --port 8000 --reload
```

Then open `http://127.0.0.1:8000/docs`.